# Experimente — Clasificare Status Final

Notebook pentru rularea și evaluarea prompturilor de clasificare a statusului final al conversațiilor.

**Task:** Clasifică statusul final al conversației în una dintre cele 5 categorii.

**Output model:**
```json
{
  "final_status": "<status>",
  "confidence": "high|medium|low",
  "reasoning": "<justificare>"
}
```

**Statusuri finale:** `resolved`, `escalated`, `partial`, `failed`, `out_of_scope`

**Structură celule:**
- [0] Setup
- [1] Config experiment
- [2] Prompt renderer + funcții utilitare
- [3] Init model
- [4] Test prompt
- [5] Rulare completă
- [6] Metrici și rezultate
- [7] Salvare experiment
- [8] Comparație experimente

In [ ]:
# [0] Setup — paths și imports
import os

os.chdir(
    r"C:\Users\Matebook 14s\Documents"
    r"\Sistem-de-monitorizare-a-interac-iunilor-voicebotilor-folosind-modele-lingvistice-mari-LLM-"
)

print("Working directory:", os.getcwd())

In [ ]:
# [0b] Imports și paths
import json
import time
import re
from pathlib import Path
from collections import defaultdict

import numpy as np
from jinja2 import Environment, FileSystemLoader
from sklearn.metrics import accuracy_score, f1_score, cohen_kappa_score, classification_report
from tabulate import tabulate

# ── Paths ──────────────────────────────────────────────────────────────────
PROMPTS_DIR  = Path("prompts/final_status")
CONFIGS_DIR  = Path("configs")
DATASET_PATH = Path("data/master_dataset_refined_180.json")
OUTPUT_DIR   = Path(r"C:\Users\Matebook 14s\Documents\Sistem-de-monitorizare-a-interac-iunilor-voicebotilor-folosind-modele-lingvistice-mari-LLM-\outputs_final_status")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Labels ─────────────────────────────────────────────────────────────────
with open(CONFIGS_DIR / "final_status_definitions.json", encoding="utf-8") as f:
    _defs = json.load(f)
STATUS_LABELS = [l["name"] for l in _defs["labels"]]
VALID_STATUSES = set(STATUS_LABELS)

# ── Dataset ────────────────────────────────────────────────────────────────
with open(DATASET_PATH, encoding="utf-8") as f:
    DATASET = json.load(f)["conversations"]

# Statistici dataset
status_counts = defaultdict(int)
for c in DATASET:
    status_counts[c["final_status"]] += 1

print(f"✓ Dataset: {len(DATASET)} conversații")
print(f"✓ Statusuri: {STATUS_LABELS}")
print(f"  Distribuție:")
for status in STATUS_LABELS:
    count = status_counts.get(status, 0)
    print(f"    {status:15s}: {count:3d} ({100*count/len(DATASET):.1f}%)")
print(f"✓ Output: {OUTPUT_DIR}")

In [ ]:
# [1] Configurație experiment — editează aici
# ═══════════════════════════════════════════════════════════

MODEL          = "openai_o3"       # openai_o3 | gemini_2.5_flash | aya_expanse_8b | rollama2_7b | roberta_encoder
LANG           = "ro"              # ro | en
PROMPT_VERSION = "v1"              # v1 | v2 | v3 | v4

# Rulează pe tot datasetul sau doar pe primele N conversații (None = tot)
MAX_CONVERSATIONS = None

# ═══════════════════════════════════════════════════════════
EXP_NAME = f"fst_{MODEL}__{LANG}__{PROMPT_VERSION}"
EXP_FILE = OUTPUT_DIR / f"exp_{EXP_NAME}.json"

conversations = DATASET[:MAX_CONVERSATIONS] if MAX_CONVERSATIONS else DATASET

print(f"Experiment : {EXP_NAME}")
print(f"Model      : {MODEL}")
print(f"Limbă      : {LANG}")
print(f"Prompt     : {LANG}_final_status_{PROMPT_VERSION}.jinja")
print(f"Dataset    : {len(conversations)} conversații")
print(f"Output     : {EXP_FILE}")

In [ ]:
# [2] Prompt renderer + parser + funcții utilitare
# ─────────────────────────────────────────────────────────────────────────

env = Environment(loader=FileSystemLoader(str(PROMPTS_DIR)))

def render_prompt(lang: str, turns: list, version: str = "v1", few_shot_examples: list = None) -> str:
    """
    Renderizează promptul Jinja2 cu conversația și opțional cu few-shot examples.
    
    Args:
        lang: 'ro' sau 'en'
        turns: lista de turn-uri
        version: 'v1', 'v2', 'v3', 'v4'
        few_shot_examples: lista de exemple pentru few-shot (doar pentru v4)
    """
    template = env.get_template(f"{lang}_final_status_{version}.jinja")
    
    if few_shot_examples:
        return template.render(conversation=turns, few_shot_examples=few_shot_examples)
    else:
        return template.render(conversation=turns)

# ── Response parser ────────────────────────────────────────────────────────
def parse_response(raw: str):
    """
    Returnează (final_status, confidence, reasoning, parse_failed).
    Gestionează JSON înfășurat în ```json ... ```.
    """
    text = raw.strip()
    text = re.sub(r"^```(?:json)?\s*", "", text)
    text = re.sub(r"\s*```$", "", text)
    try:
        data = json.loads(text)
        status = data.get("final_status")
        # Validare
        if status not in VALID_STATUSES:
            return status, data.get("confidence"), data.get("reasoning"), True
        return (
            status,
            data.get("confidence"),
            data.get("reasoning"),
            False,
        )
    except json.JSONDecodeError:
        return None, None, None, True

# ── Helper: eticheta din dataset ──────────────────────────────────────────
def get_dataset_label(conv: dict):
    """Returnează final_status din dataset."""
    return conv.get("final_status")

# ── Metrici ────────────────────────────────────────────────────────────────
def compute_metrics(results: list):
    """Metrici pentru clasificarea statusului final (5 clase)."""
    valid = [r for r in results if not r["parse_failed"] and r["predicted_status"] is not None]
    if not valid:
        return {}
    y_true = [r["dataset_status"] for r in valid]
    y_pred = [r["predicted_status"] for r in valid]
    return {
        "accuracy":    round(accuracy_score(y_true, y_pred), 4),
        "macro_f1":    round(f1_score(y_true, y_pred, average="macro", labels=STATUS_LABELS, zero_division=0), 4),
        "weighted_f1": round(f1_score(y_true, y_pred, average="weighted", labels=STATUS_LABELS, zero_division=0), 4),
        "kappa":       round(cohen_kappa_score(y_true, y_pred), 4),
        "n_valid":     len(valid),
    }

# ── Tabel metrici sumar ────────────────────────────────────────────────────
def print_metrics_summary(results: list, exp_name: str):
    fails = sum(1 for r in results if r["parse_failed"])
    total = len(results)
    lats  = [r["latency_ms"] for r in results if r["latency_ms"] > 0]

    m = compute_metrics(results)

    rows = [
        ["Experiment",        exp_name],
        ["Accuracy",          f"{m.get('accuracy', 0)*100:.1f}%"],
        ["Macro F1",          f"{m.get('macro_f1', 0)*100:.1f}%"],
        ["Weighted F1",       f"{m.get('weighted_f1', 0)*100:.1f}%"],
        ["Cohen's Kappa",     f"{m.get('kappa', 0):.3f}"],
        ["Parse failures",    f"{fails}/{total}  ({fails/total*100:.1f}%)"],
        ["Latență medie",     f"{np.mean(lats):.0f}ms" if lats else "—"],
        ["Latență p95",       f"{np.percentile(lats, 95):.0f}ms" if lats else "—"],
        ["N valid",           m.get('n_valid', 0)],
    ]
    print(tabulate(rows, tablefmt="simple", headers=["Metrică", "Valoare"]))
    return {**m, "n_failed": fails}

# ── Tabel predicții ────────────────────────────────────────────────────────
def print_predictions_table(results: list):
    rows = []
    for r in results:
        if r["parse_failed"]:
            status = "FAIL"
        elif r["predicted_status"] == r["dataset_status"]:
            status = f"✓  {r['predicted_status']}"
        else:
            status = f"✗  {r['predicted_status']}  (real: {r['dataset_status']})"
        rows.append([
            r["conversation_id"],
            r["dataset_status"],
            status,
            r.get("confidence", "—"),
            f"{r['latency_ms']:.0f}ms",
        ])
    print(tabulate(rows,
                   headers=["conv_id", "dataset_label", "predicție", "confidence", "latență"],
                   tablefmt="rounded_outline"))

# ── Top erori ──────────────────────────────────────────────────────────────
def print_top_errors(results: list):
    errors = defaultdict(lambda: defaultdict(int))
    for r in results:
        if r["parse_failed"]:
            continue
        true_status = r["dataset_status"]
        pred_status = r["predicted_status"]
        if true_status != pred_status:
            errors[true_status][pred_status] += 1

    rows = [
        [true, pred, cnt]
        for true, preds in errors.items()
        for pred, cnt in preds.items()
    ]
    rows.sort(key=lambda r: -r[2])
    if rows:
        print(tabulate(rows[:10],
                       headers=["Status real", "Prezis ca", "# greșeli"],
                       tablefmt="simple"))
    else:
        print("  0 erori.")

# ── Classification report ─────────────────────────────────────────────────
def print_classification_report(results: list):
    valid = [r for r in results if not r["parse_failed"] and r["predicted_status"] is not None]
    if not valid:
        print("Nu există predicții valide pentru raport.")
        return
    y_true = [r["dataset_status"] for r in valid]
    y_pred = [r["predicted_status"] for r in valid]
    print(classification_report(y_true, y_pred, labels=STATUS_LABELS, zero_division=0))

print("✓ Funcții utilitare încărcate.")

In [ ]:
# [3] Inițializează modelul selectat în [1]
# ─────────────────────────────────────────────────────────────────────────
from dotenv import load_dotenv
load_dotenv()

if MODEL == "openai_o3":
    from openai import OpenAI
    _client = OpenAI()

    def call_model(prompt: str) -> str:
        r = _client.chat.completions.create(
            model="o3",
            messages=[{"role": "user", "content": prompt}],
        )
        return r.choices[0].message.content

elif MODEL == "gemini_2.5_flash":
    import os
    from google.genai import Client
    _client = Client(api_key=os.environ["GEMINI_API_KEY"])

    def call_model(prompt: str) -> str:
        response = _client.models.generate_content(
            model="gemini-2.5-flash",
            contents=prompt,
        )
        return response.text

elif MODEL == "aya_expanse_8b":
    import ollama as _ollama

    def call_model(prompt: str) -> str:
        r = _ollama.chat(
            model="aya-expanse:8b",
            messages=[{"role": "user", "content": prompt}],
        )
        return r["message"]["content"]

elif MODEL == "rollama2_7b":
    import ollama as _ollama

    def call_model(prompt: str) -> str:
        r = _ollama.chat(
            model="rollama2-instruct:7b",
            messages=[{"role": "user", "content": prompt}],
        )
        return r["message"]["content"]

elif MODEL == "roberta_encoder":
    from transformers import pipeline
    _classifier = pipeline(
        "text-classification",
        model="readerbench/RoBERT-base",
        return_all_scores=True,
    )

    def call_model(prompt: str) -> str:
        # Pentru encoder models, trebuie să extragi conversația și să clasific
        # Aici e un placeholder - adaptează conform nevoilor
        results = _classifier(prompt[:512])  # Truncate la 512 tokens
        # Returnează ca JSON
        top_pred = max(results[0], key=lambda x: x['score'])
        return json.dumps({
            "final_status": top_pred['label'],
            "confidence": "high" if top_pred['score'] > 0.8 else "medium" if top_pred['score'] > 0.5 else "low",
            "reasoning": f"Classification score: {top_pred['score']:.3f}"
        })

else:
    raise ValueError(f"Model necunoscut: {MODEL}")

print(f"✓ Model inițializat: {MODEL}")

In [ ]:
# [4] Test prompt pe o conversație
# ─────────────────────────────────────────────────────────────────────────

# Încarcă few-shot examples pentru v4
few_shot_examples = []
if PROMPT_VERSION == "v4":
    few_shot_file = "few_shot_examples_final_status.json" if LANG == "ro" else "few_shot_examples_final_status_en.json"
    try:
        with open(CONFIGS_DIR / few_shot_file, encoding="utf-8") as f:
            few_shot_data = json.load(f)
            few_shot_examples = few_shot_data.get("examples", [])
        print(f"✓ Încărcat {len(few_shot_examples)} exemple few-shot pentru {LANG} {PROMPT_VERSION}")
    except FileNotFoundError:
        print(f"⚠ Nu s-a găsit {few_shot_file} - se continuă fără few-shot examples")

sample_conv = conversations[0]
ds_status = get_dataset_label(sample_conv)

sample_prompt = render_prompt(LANG, sample_conv["turns"], PROMPT_VERSION, few_shot_examples)

print(f"── Conversație: {sample_conv['conversation_id']}")
print(f"── dataset final_status: {ds_status}")
print("─" * 60)
print(sample_prompt)
print("─" * 60)

raw = call_model(sample_prompt)
print("RAW:", repr(raw))
parsed_status, conf, reason, failed = parse_response(raw)
print(f"PARSED: status={parsed_status}, confidence={conf}, failed={failed}")
print(f"REASONING: {reason}")

In [ ]:
# [5] Rulare completă pe dataset
# ─────────────────────────────────────────────────────────────────────────

# Încarcă few-shot examples pentru v4
few_shot_examples = []
if PROMPT_VERSION == "v4":
    few_shot_file = "few_shot_examples_final_status.json" if LANG == "ro" else "few_shot_examples_final_status_en.json"
    try:
        with open(CONFIGS_DIR / few_shot_file, encoding="utf-8") as f:
            few_shot_data = json.load(f)
            few_shot_examples = few_shot_data.get("examples", [])
        print(f"✓ Încărcat {len(few_shot_examples)} exemple few-shot")
    except FileNotFoundError:
        print(f"⚠ {few_shot_file} nu a fost găsit - se continuă fără few-shot examples")

results = []
total = len(conversations)

print(f"Rulare experiment: {EXP_NAME}")
print(f"Dataset: {total} conversații\n")

for i, conv in enumerate(conversations):
    conv_id   = conv["conversation_id"]
    ds_status = get_dataset_label(conv)
    turns     = conv["turns"]

    prompt = render_prompt(LANG, turns, PROMPT_VERSION, few_shot_examples)

    start = time.perf_counter()
    try:
        raw = call_model(prompt)
        pred_status, confidence, reasoning, parse_failed = parse_response(raw)
        elapsed_ms = (time.perf_counter() - start) * 1000

        results.append({
            "conversation_id": conv_id,
            "dataset_status": ds_status,
            "predicted_status": pred_status,
            "confidence": confidence,
            "reasoning": reasoning,
            "raw_response": raw,
            "parse_failed": parse_failed,
            "latency_ms": elapsed_ms,
        })

        # Progress indicator
        if parse_failed:
            print(f"[{i+1:3d}/{total}] ✗ {conv_id} — PARSE FAILED")
        elif pred_status == ds_status:
            print(f"[{i+1:3d}/{total}] ✓ {conv_id}")
        else:
            print(f"[{i+1:3d}/{total}] ✗ {conv_id} — pred:{pred_status} vs real:{ds_status}")

    except Exception as e:
        elapsed_ms = (time.perf_counter() - start) * 1000
        results.append({
            "conversation_id": conv_id,
            "dataset_status": ds_status,
            "predicted_status": None,
            "confidence": None,
            "reasoning": None,
            "raw_response": "",
            "parse_failed": True,
            "latency_ms": elapsed_ms,
        })
        print(f"[{i+1:3d}/{total}] ✗ {conv_id} — EXCEPTION: {e}")

    # Batch progress
    if (i + 1) % 20 == 0:
        print(f"\n── Progres: {i+1}/{total} ({100*(i+1)/total:.1f}%) ──\n")

print(f"\n✓ Experiment finalizat: {len(results)} conversații procesate")

In [ ]:
# [6] Metrici și analiză rezultate
# ─────────────────────────────────────────────────────────────────────────

print("═" * 80)
print(f"REZULTATE EXPERIMENT: {EXP_NAME}")
print("═" * 80)
print()

# Metrici sumar
print("📊 METRICI SUMAR")
print("─" * 80)
metrics = print_metrics_summary(results, EXP_NAME)
print()

# Classification report
print("📋 CLASSIFICATION REPORT")
print("─" * 80)
print_classification_report(results)
print()

# Top erori
print("❌ TOP ERORI DE CLASIFICARE")
print("─" * 80)
print_top_errors(results)
print()

# Tabel predicții (primele 20)
print("📝 PREDICȚII (primele 20)")
print("─" * 80)
print_predictions_table(results[:20])
print()

print("═" * 80)

In [ ]:
# [7] Salvare rezultate experiment
# ─────────────────────────────────────────────────────────────────────────

experiment_data = {
    "experiment_name": EXP_NAME,
    "model": MODEL,
    "language": LANG,
    "prompt_version": PROMPT_VERSION,
    "dataset_size": len(conversations),
    "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
    "metrics": compute_metrics(results),
    "results": results,
}

with open(EXP_FILE, "w", encoding="utf-8") as f:
    json.dump(experiment_data, f, ensure_ascii=False, indent=2)

print(f"✓ Experiment salvat: {EXP_FILE}")
print(f"  Model: {MODEL}")
print(f"  Limbă: {LANG}")
print(f"  Prompt: {PROMPT_VERSION}")
print(f"  Accuracy: {metrics['accuracy']*100:.1f}%")
print(f"  Macro F1: {metrics['macro_f1']*100:.1f}%")

In [ ]:
# [8] Comparație experimente
# ─────────────────────────────────────────────────────────────────────────

# Încarcă toate experimentele salvate
experiment_files = sorted(OUTPUT_DIR.glob("exp_fst_*.json"))

if not experiment_files:
    print("Nu există experimente salvate.")
else:
    comparisons = []
    for exp_file in experiment_files:
        with open(exp_file, encoding="utf-8") as f:
            exp_data = json.load(f)
        
        comparisons.append([
            exp_data["experiment_name"],
            exp_data["model"],
            exp_data["language"],
            exp_data["prompt_version"],
            f"{exp_data['metrics'].get('accuracy', 0)*100:.1f}%",
            f"{exp_data['metrics'].get('macro_f1', 0)*100:.1f}%",
            f"{exp_data['metrics'].get('weighted_f1', 0)*100:.1f}%",
            exp_data['metrics'].get('n_failed', 0),
        ])
    
    # Sortează după Macro F1 descrescător
    comparisons.sort(key=lambda x: float(x[5].rstrip('%')), reverse=True)
    
    print("═" * 120)
    print("COMPARAȚIE EXPERIMENTE FINAL_STATUS")
    print("═" * 120)
    print(tabulate(
        comparisons,
        headers=["Experiment", "Model", "Lang", "Ver", "Acc", "Macro F1", "Weighted F1", "Fails"],
        tablefmt="rounded_outline"
    ))
    print("═" * 120)